Package install

In [ ]:
%pip install scikit-learn xgboost lightgbm seaborn matplotlib

Library Import

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor



Data Preview

In [ ]:
df = pd.read_csv("quebec_housing_sales_cleaned.csv")

print(df.info())
print(df.describe())
print(df.head())


Encode Category DaTA

In [ ]:
from sklearn.preprocessing import LabelEncoder
cat_cols = df.select_dtypes(include=['object']).columns
for col in cat_cols:
    df[col] = LabelEncoder().fit_transform(df[col])

df.fillna(0, inplace=True)
df.describe()

Split

In [ ]:
X = df.drop(columns=['sale_price'])
y = df['sale_price']

rf_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('rf', RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42))
])

xgb_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('xgb', XGBRegressor(n_estimators=200, max_depth=6, learning_rate=0.1, random_state=42))
])

# Define K-Fold
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Cross-validated predictions
rf_pred = cross_val_predict(rf_pipeline, X, y, cv=kf)
xgb_pred = cross_val_predict(xgb_pipeline, X, y, cv=kf)

# Metrics
rf_rmse = np.sqrt(mean_squared_error(y, rf_pred))
rf_r2   = r2_score(y, rf_pred)

xgb_rmse = np.sqrt(mean_squared_error(y, xgb_pred))
xgb_r2   = r2_score(y, xgb_pred)

print("Random Forest -> RMSE:", rf_rmse, "R²:", rf_r2)
print("XGBoost       -> RMSE:", xgb_rmse, "R²:", xgb_r2)



Comparisison of models

In [ ]:
plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
sns.scatterplot(x=y, y=y-rf_pred, color='blue')
plt.axhline(0, color='red', linestyle='--')
plt.title("Random Forest Residuals")
plt.xlabel("Actual Sale Price")
plt.ylabel("Residuals")

plt.subplot(1,2,2)
sns.scatterplot(x=y, y=y-xgb_pred, color='orange')
plt.axhline(0, color='red', linestyle='--')
plt.title("XGBoost Residuals")
plt.xlabel("Actual Sale Price")
plt.ylabel("Residuals")

plt.tight_layout()
plt.savefig("residuals_comparison.png", dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
Visualization of feature importance for Random Forest and XGBoost models:

In [ ]:
rf_scores = cross_val_score(rf_pipeline, X, y, cv=kf, scoring='neg_mean_squared_error')
xgb_scores = cross_val_score(xgb_pipeline, X, y, cv=kf, scoring='neg_mean_squared_error')

rf_rmse = np.sqrt(-rf_scores)
xgb_rmse = np.sqrt(-xgb_scores)

plt.figure(figsize=(8,6))
plt.plot(range(1, len(rf_rmse)+1), rf_rmse, marker='o', label='Random Forest', color='blue')
plt.plot(range(1, len(xgb_rmse)+1), xgb_rmse, marker='s', label='XGBoost', color='orange')

plt.title("K-Fold RMSE Comparison")
plt.xlabel("Fold")
plt.ylabel("RMSE")
plt.legend()
plt.tight_layout()
plt.savefig("kfold_lineplot.png", dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
plt.figure(figsize=(6,6))
sns.boxplot(data=[rf_rmse, xgb_rmse])
plt.xticks([0,1], ['Random Forest','XGBoost'])
plt.title("RMSE Distribution Across Folds")
plt.ylabel("RMSE")
plt.tight_layout()
plt.savefig("kfold_boxplot.png", dpi=300, bbox_inches='tight')
plt.show()


Prediction Function

In [ ]:
def predict_price(input_dict, rf_model, xgb_model):

    new_input = pd.DataFrame([input_dict])

    rf_price = rf_model.predict(new_input)[0]
    xgb_price = xgb_model.predict(new_input)[0]

    return {
        "Random Forest Prediction": rf_price,
        "XGBoost Prediction": xgb_price
    }


Checking Accuracy

In [ ]:

rf_pipeline.fit(X, y)
xgb_pipeline.fit(X, y)

new_house = {
    'city': 6,
    'neighborhood': 11,
    'property_type': 3,
    'bedrooms': 3,
    'bathrooms': 2,
    'living_area_sqft': 1160,
    'lot_size_sqft': 2304,
    'year_built': 2022,
    'garage': 1,
    'basement': 0,
    'sale_year': 2024,
}
predictions = predict_price(new_house, rf_pipeline, xgb_pipeline)
print(predictions)
